In [1]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install sentence-transformers pandas

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --- 1. 데이터 로드 ---
try:
    # 게임 정보 데이터 로드
    games_df = pd.read_csv("/content/drive/MyDrive/2025Bigdata/data/game_dataset/steam_data.csv")
    # 리뷰 데이터 로드
    reviews_df = pd.read_csv("/content/drive/MyDrive/2025Bigdata/data/game_dataset/steam_top_2_reviews.csv")
    print("csv 파일 로드 성공")
except FileNotFoundError:
    print("오류: csv 파일이 경로에 없습니다. 파일 업로드를 확인해주세요.")
    exit()

# --- 2. 리뷰 데이터 전처리 및 병합 ---
# 게임별로 여러 개의 리뷰가 있을 수 있으므로, 하나로 합칩니다.
print("리뷰 데이터 병합 중...")

# 리뷰 텍스트가 비어있는 경우 빈 문자열로 처리
reviews_df['review_text'] = reviews_df['review_text'].fillna('')

# 같은 게임(app_id)의 리뷰들을 공백으로 구분하여 하나로 합침 (최대 길이 제한을 위해 일부만 사용할 수도 있음)
# 여기서는 각 게임별 상위 N개의 리뷰만 합치거나 전체를 합치는 방식을 사용합니다.
grouped_reviews = reviews_df.groupby('app_id')['review_text'].apply(lambda x: ' '.join(x)).reset_index()

# 게임 데이터(games_df)와 리뷰 데이터(grouped_reviews)를 병합 (Left Join: 리뷰가 없는 게임도 유지)
# games_df의 'appid'와 grouped_reviews의 'app_id'를 기준으로 병합
merged_df = pd.merge(games_df, grouped_reviews, left_on='appid', right_on='app_id', how='left')

# 병합 후 'review_text'가 NaN인 경우(리뷰가 없는 게임) 빈 문자열로 채움
merged_df['review_text'] = merged_df['review_text'].fillna('')

print(f"데이터 병합 완료. 총 게임 수: {len(merged_df)}")

# --- 3. '종합 텍스트' 생성 함수 (리뷰 포함) ---
def create_meta_text(row):
    name = str(row.get('original_name', ''))
    genres = str(row.get('genres', '')).replace('|', ' ').replace('[', '').replace(']', '').replace("'", "")
    tags = str(row.get('tags', '')).replace('[', '').replace(']', '').replace("'", "")
    desc = str(row.get('short_description', ''))
    review = str(row.get('review_text', ''))

    # 1. 텍스트 정제 (줄바꿈 제거 등)
    review = review.replace('\n', ' ').strip()

    # 2. 리뷰 길이 제한 (토큰 제한 고려, 너무 길면 앞부분 500자만 사용)
    # 모델의 입력 한계(보통 512토큰)를 고려하여 리뷰 비중 조절이 필요합니다.
    if len(review) > 500:
        review = review[:500]

    # 텍스트가 없는 게임을 위해 이름이라도 반환
    if not genres and not tags and not desc and not review:
        return name

    # 3. 텍스트 합치기 (중요도 순서: 이름 > 장르/태그 > 설명 > 리뷰)
    return f"{name} {genres} {tags} {desc} {review}"

print("게임별 '종합 텍스트' 생성 중...")
merged_df['meta_text'] = merged_df.apply(create_meta_text, axis=1)

# --- 4. SentenceTransformer 모델 로드 ---
model_name = 'paraphrase-mpnet-base-v2'
model = SentenceTransformer(model_name)
print(f"'{model_name}' 모델 로드 성공")

# --- 5. 텍스트를 벡터로 변환 (인코딩) ---
print(f"{len(merged_df)}개의 게임 설명을 벡터로 변환합니다... (시간이 걸릴 수 있습니다)")

# meta_text 리스트 추출
game_texts = merged_df['meta_text'].tolist()

# 임베딩 생성
game_text_vectors = model.encode(game_texts, show_progress_bar=True)
print(f"벡터 변환 완료. 벡터 형태: {game_text_vectors.shape}")

# --- 6. 벡터 DB 파일로 저장 ---
output_filename = "game_text_vectors.npy"
np.save(output_filename, game_text_vectors)

print(f"\n✅ 성공! '{output_filename}' 파일이 생성되었습니다.")
print("이 파일을 다운로드하여 사용하세요.")

# (선택 사항) Colab에서 바로 다운로드
from google.colab import files
files.download(output_filename)

csv 파일 로드 성공
리뷰 데이터 병합 중...
데이터 병합 완료. 총 게임 수: 10000
게임별 '종합 텍스트' 생성 중...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

'paraphrase-mpnet-base-v2' 모델 로드 성공
10000개의 게임 설명을 벡터로 변환합니다... (시간이 걸릴 수 있습니다)


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

벡터 변환 완료. 벡터 형태: (10000, 768)

✅ 성공! 'game_text_vectors.npy' 파일이 생성되었습니다.
이 파일을 다운로드하여 사용하세요.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>